In [5]:
!pip install wandb gensim sentence-transformers transformers sentencepiece faiss-cpu peft -q
!pip uninstall -y hf_xet -q

import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import os
os.environ["WANDB_SILENT"] = "true"

from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
w_key = user_secrets.get_secret("WANDB_Key")
wandb.login(key=w_key)

import torch
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration, pipeline, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, TaskType
import time


# retry model download if network issue occurs
def load_with_retry(load_fn, *args, retries=5, delay=8, **kwargs):
    for attempt in range(retries):
        try:
            return load_fn(*args, **kwargs)
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"download failed (attempt {attempt+1}/{retries}), retrying in {delay}s...")
            time.sleep(delay)
OPTS   = ['A', 'B', 'C', 'D', 'E']
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| device:", device)
train.head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 41.5 MB/s eta 0:00:00a 0:00:01
train: (2000, 8) | test: (500, 7) | device: cuda


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [6]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

=== null counts ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== answer distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

=== prompt length (words) ===
count    2000.00
mean       18.15
std         6.78
min         3.00
25%        14.00
50%        17.00
75%        22.00
max        51.00
Name: prompt, dtype: float64

=== avg option length (words) ===
  A: 26.1
  B: 26.5
  C: 26.5
  D: 26.0
  E: 26.2


In [7]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

unique vocab in cleaned prompts: 859
sample — original : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
sample — cleaned  : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options


In [8]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

LETTER2ID = {o: i for i, o in enumerate(OPTS)}
ID2LETTER  = {i: o for o, i in LETTER2ID.items()}

full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])

# char n-grams catch morphology/phrasing patterns word-level tfidf misses
word_tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=25000, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=15000, sublinear_tf=True)
word_tfidf.fit(full_corpus)
char_tfidf.fit(full_corpus)
print(f"word tfidf vocab: {len(word_tfidf.vocabulary_)} | char tfidf vocab: {len(char_tfidf.vocabulary_)}")

def build_row_features(row):
    p_word = word_tfidf.transform([row['prompt']])
    p_char = char_tfidf.transform([row['prompt']])
    feats = []
    for o in OPTS:
        o_word = word_tfidf.transform([row[o]])
        o_char = char_tfidf.transform([row[o]])
        word_sim = cosine_similarity(p_word, o_word)[0][0]
        char_sim = cosine_similarity(p_char, o_char)[0][0]
        len_ratio = len(row[o].split()) / max(len(row['prompt'].split()), 1)
        p_tokens = set(clean_text(row['prompt']).split())
        o_tokens = set(clean_text(row[o]).split())
        overlap = len(p_tokens & o_tokens) / max(len(p_tokens | o_tokens), 1)  # jaccard
        feats.extend([word_sim, char_sim, len_ratio, overlap])
    return feats

print("building features for all train rows...")
X = np.array([build_row_features(row) for _, row in train.iterrows()])
y = train['answer'].map(LETTER2ID).values

# out-of-fold. each rows prediction comes from a fold that never trained on it
# reportS MAP@3 on the FULL 2000 rows without leakage
oof_probs = np.zeros((len(train), 5))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    clf = LogisticRegression(max_iter=2000, C=2.0, random_state=42)
    clf.fit(X[tr_idx], y[tr_idx])
    oof_probs[va_idx] = clf.predict_proba(X[va_idx])
    print(f"fold {fold+1}/5 done")

# final classifier trained on all data, used for actual test predictions
tfidf_clf = LogisticRegression(max_iter=2000, C=2.0, random_state=42)
tfidf_clf.fit(X, y)

def rank_tfidf(row):
    feats = np.array(build_row_features(row)).reshape(1, -1)
    probs = tfidf_clf.predict_proba(feats)[0]
    scores = {ID2LETTER[i]: probs[i] for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

def top1_accuracy(actuals, predictions):
    return round(accuracy_score(actuals, [p[0] for p in predictions]), 4)

def top1_f1(actuals, predictions):
    return round(f1_score(actuals, [p[0] for p in predictions], average='macro'), 4)

actuals = train['answer'].tolist()
oof_ranked = []
for probs in oof_probs:
    scores = {ID2LETTER[i]: probs[i] for i in range(5)}
    oof_ranked.append(sorted(scores, key=lambda x: scores[x], reverse=True)[:3])

map3_tfidf = round(mapk(actuals, oof_ranked), 4)
acc_tfidf  = top1_accuracy(actuals, oof_ranked)
f1_tfidf   = top1_f1(actuals, oof_ranked)

print(f"\ntfidf supervised (5-fold OOF)  MAP@3 : {map3_tfidf}")
print(f"tfidf — acc: {acc_tfidf}, f1: {f1_tfidf}")

run = wandb.init(project="23f1000362-t22026", name="model1-fromscratch-tfidf-supervised",
                  config={"features": "word+char tfidf cosine, len_ratio, jaccard", "cv": "5fold_oof", "C": 2.0},
                  tags=["milestone-5", "from-scratch", "tfidf"])
wandb.log({"map3": map3_tfidf, "accuracy": acc_tfidf, "f1": f1_tfidf})
wandb.finish()

print("\ncell 4 complete - tfidf supervised (OOF) trained and logged")

word tfidf vocab: 10915 | char tfidf vocab: 15000
building features for all train rows...
fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done

tfidf supervised (5-fold OOF)  MAP@3 : 0.6427
tfidf — acc: 0.496, f1: 0.4939

cell 4 complete - tfidf supervised (OOF) trained and logged


In [ ]:
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))

w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
print(f"w2v vocab: {len(w2v_model.wv)}")

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1); v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

w2v_preds = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v = round(mapk(actuals, w2v_preds), 4)
acc_w2v  = top1_accuracy(actuals, w2v_preds)
f1_w2v   = top1_f1(actuals, w2v_preds)

print(f"word2vec cosine  MAP@3 : {map3_w2v}")
print(f"w2v — acc: {acc_w2v}, f1: {f1_w2v}")

run = wandb.init(project="23f1000362-t22026", name="model2-fromscratch-word2vec", tags=["milestone-5", "from-scratch", "word2vec"])
wandb.log({"map3": map3_w2v, "accuracy": acc_w2v, "f1": f1_w2v})
wandb.finish()

print("\ncell 5 complete - word2vec trained and logged")

In [ ]:
bert_tokenizer = load_with_retry(AutoTokenizer.from_pretrained, "roberta-base")
bert_model     = load_with_retry(AutoModel.from_pretrained, "roberta-base", output_attentions=True).to(device)
bert_model.eval()

sample_inputs = bert_tokenizer(train['prompt'].iloc[0], return_tensors="pt").to(device)
with torch.no_grad():
    sample_out = bert_model(**sample_inputs)
print(f"attention layers: {len(sample_out.attentions)}, heads per layer: {sample_out.attentions[0].shape[1]}")

def get_roberta_embedding(text):
    inputs = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
    with torch.no_grad():
        out = bert_model(**inputs)
    token_embs = out.last_hidden_state
    mask = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
    # mean pooling over token embeddings
    summed = torch.sum(token_embs * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return (summed / counts).squeeze(0).cpu().numpy()

def rank_roberta(row):
    pv = get_roberta_embedding(row['prompt']).reshape(1, -1)
    sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

sample_roberta = train.sample(n=200, random_state=42)
roberta_preds  = [rank_roberta(row)[:3] for _, row in sample_roberta.iterrows()]
map3_roberta   = round(mapk(sample_roberta['answer'].tolist(), roberta_preds), 4)
acc_roberta    = top1_accuracy(sample_roberta['answer'].tolist(), roberta_preds)
f1_roberta     = top1_f1(sample_roberta['answer'].tolist(), roberta_preds)

print(f"roberta cosine (200)  MAP@3 : {map3_roberta}")
print(f"roberta — acc: {acc_roberta}, f1: {f1_roberta}")

run = wandb.init(project="23f1000362-t22026", name="model3-pretrained-roberta-frozen", tags=["milestone-5", "pretrained", "roberta"])
wandb.log({"map3": map3_roberta, "accuracy": acc_roberta, "f1": f1_roberta})
wandb.finish()

print("\ncell 6 complete - roberta cosine trained and logged")

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def format_mcq(row):
    return (f"{row['prompt']} "
            f"A) {row['A']} B) {row['B']} C) {row['C']} D) {row['D']} E) {row['E']}")

# small val split only for tracking loss during training
loss_tr_idx, loss_va_idx = train_test_split(
    train.index,
    test_size=0.1,
    stratify=train['answer'],
    random_state=42
)
loss_train_df = train.loc[loss_tr_idx].reset_index(drop=True)
loss_val_df   = train.loc[loss_va_idx].reset_index(drop=True)

cls_tok = bert_tokenizer
cls_collator = DataCollatorWithPadding(cls_tok)

def encode_split(df):
    inputs = [format_mcq(row) for _, row in df.iterrows()]
    labels = [LETTER2ID[a] for a in df['answer']]
    enc = cls_tok(inputs, truncation=True, max_length=256, padding=False)
    enc['labels'] = labels
    return Dataset.from_dict(enc)

hf_loss_train = encode_split(loss_train_df)
hf_loss_val   = encode_split(loss_val_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

loss_base = load_with_retry(
    AutoModelForSequenceClassification.from_pretrained,
    "roberta-base",
    num_labels=5
)

# apply lora instead of finetuning all model params
loss_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)
loss_model = get_peft_model(loss_base, loss_cfg).to(device)

loss_args = TrainingArguments(
    output_dir="/kaggle/working/lora_roberta_losscurve",
    num_train_epochs=6,                 # train for 6 full passes
    per_device_train_batch_size=16,     # fits in kaggle gpu memory
    gradient_accumulation_steps=2,      # effective batch = 32
    learning_rate=2e-4,                 # common lr for lora finetuning
    warmup_steps=50,                    # warmup for stable training
    fp16=True,                          # faster training on gpu
    logging_steps=20,
    eval_strategy="epoch",              # eval after each epoch
    save_strategy="no",                 # no checkpoints needed
    report_to="wandb",                  # log train/eval metrics
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=42
)

wandb.init(
    project="23f1000362-t22026",
    name="model4-lora-roberta-losscurve",
    config={"lora_r": 16, "lora_alpha": 32, "lr": 2e-4, "epochs": 6},
    tags=["milestone-5", "lora", "roberta", "loss-curve"]
)

loss_trainer = Trainer(
    model=loss_model,
    args=loss_args,
    train_dataset=hf_loss_train,
    eval_dataset=hf_loss_val,
    data_collator=cls_collator,
    processing_class=cls_tok,
    compute_metrics=compute_metrics
)

print("training lora roberta WITH train+val loss logging to wandb...")
loss_trainer.train()
wandb.finish()
print("loss curve run complete — check WandB dashboard for train/eval loss charts")

# train final model on complete train set
train_inputs = [format_mcq(row) for _, row in train.iterrows()]
train_labels = [LETTER2ID[a] for a in train['answer']]
cls_enc = cls_tok(train_inputs, truncation=True, max_length=256, padding=False)
cls_enc['labels'] = train_labels
hf_cls = Dataset.from_dict(cls_enc)

def train_lora_clean(seed):
    base = load_with_retry(
        AutoModelForSequenceClassification.from_pretrained,
        "roberta-base",
        num_labels=5
    )

    cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                               # rank of lora adapters
    lora_alpha=32,                      # scaling factor
    target_modules=["query", "value"],  # apply lora only here
    lora_dropout=0.1,
    bias="none"
    )

    model = get_peft_model(base, cfg).to(device)

    args = TrainingArguments(
        output_dir=f"/kaggle/working/lora_clean_seed{seed}",
        num_train_epochs=6,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_steps=50,
        fp16=True,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
        seed=seed
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=hf_cls,
        data_collator=cls_collator,
        processing_class=cls_tok
    )

    trainer.train()
    model.eval()
    return model

# train same model with diff random seeds
print("training clean seed=42...")
lora_model = train_lora_clean(42)

print("training clean seed=123...")
lora_model_2 = train_lora_clean(123)

print("training clean seed=7...")
lora_model_3 = train_lora_clean(7)

lora_models = [lora_model, lora_model_2, lora_model_3]

def get_lora_logits(model, row):
    inputs = cls_tok(
        format_mcq(row),
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits[0]

    return logits.cpu().numpy()

def rank_lora(row):
    logits = get_lora_logits(lora_model, row)
    scores = {ID2LETTER[i]: logits[i] for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

# simple avg of logits from all models
def rank_lora_ensemble_avg(row):
    all_logits = np.array([get_lora_logits(m, row) for m in lora_models])
    avg_logits = all_logits.mean(axis=0)
    scores = {ID2LETTER[i]: avg_logits[i] for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

lora_preds = [rank_lora(row)[:3] for _, row in train.iterrows()]
map3_lora = round(mapk(actuals, lora_preds), 4)

ens_preds = [rank_lora_ensemble_avg(row)[:3] for _, row in train.iterrows()]
map3_ens = round(mapk(actuals, ens_preds), 4)

print(f"lora single (seed42)        MAP@3 : {map3_lora}")
print(f"lora clean 3-seed avg ens   MAP@3 : {map3_ens}")

def _quick_acc(model, sample_df):
    correct = 0
    for _, row in sample_df.iterrows():
        ins = cls_tok(
            format_mcq(row),
            return_tensors="pt",
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            lg = model(**ins).logits[0]

        if ID2LETTER[int(torch.argmax(lg))] == row['answer']:
            correct += 1

    return correct / len(sample_df)

# estimate model quality to use as ensemble weights
_acc_sample = train.sample(n=300, random_state=99)

# use sample accuracy as ensemble weights
lora_weights = [_quick_acc(m, _acc_sample) for m in lora_models]

print("per-seed sample accuracy (ensemble weights):",
      [round(w, 4) for w in lora_weights])

# avg probabilities using weighted contribution from each model
def rank_lora_ensemble_weighted(row):
    all_logits = np.array([get_lora_logits(m, row) for m in lora_models])

    # convert logits to probs before weighted averaging
    probs = np.exp(all_logits - all_logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)

    w = np.array(lora_weights) / sum(lora_weights)
    avg = np.average(probs, axis=0, weights=w)

    scores = {ID2LETTER[i]: avg[i] for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

ens_wt_preds = [rank_lora_ensemble_weighted(row)[:3] for _, row in train.iterrows()]
map3_ens_wt = round(mapk(actuals, ens_wt_preds), 4)

print(f"lora weighted (prob-avg) ens MAP@3 : {map3_ens_wt}")

run = wandb.init(
    project="23f1000362-t22026",
    name="model4-lora-roberta-clean-3seed-avg",
    config={
        "lora_r": 16,
        "lora_alpha": 32,
        "lr": 2e-4,
        "epochs": 6,
        "seeds": [42, 123, 7]
    },
    tags=["milestone-5", "lora", "roberta", "clean-ensemble"]
)

wandb.log({
    "map3_single": map3_lora,
    "map3_ensemble": map3_ens,
    "map3_ensemble_weighted": map3_ens_wt
})

wandb.finish()

In [ ]:
from sklearn.model_selection import train_test_split

deberta_tok  = load_with_retry(AutoTokenizer.from_pretrained, "microsoft/deberta-v3-base")
deberta_base = load_with_retry(
    AutoModelForSequenceClassification.from_pretrained,
    "microsoft/deberta-v3-base",
    num_labels=5
)

# keep a small val split to monitor training
tr_df, val_df = train_test_split(
    train,
    test_size=0.1,
    stratify=train["answer"],
    random_state=42
)

def encode_deberta(df):
    inputs = [format_mcq(row) for _, row in df.iterrows()]
    labels = [LETTER2ID[a] for a in df["answer"]]
    enc = deberta_tok(
        inputs,
        truncation=True,
        max_length=256,
        padding=False
    )
    enc["labels"] = labels
    return Dataset.from_dict(enc)

hf_deberta_train = encode_deberta(tr_df)
hf_deberta_val   = encode_deberta(val_df)

# deberta uses query_proj/value_proj instead of query/value
lora_cfg_deberta = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                               # rank of lora adapters
    lora_alpha=32,                      # scaling factor
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none"
)

lora_deberta = get_peft_model(deberta_base, lora_cfg_deberta).to(device)
lora_deberta.print_trainable_parameters()

deberta_collator = DataCollatorWithPadding(deberta_tok)

args_deberta = TrainingArguments(
    output_dir="/kaggle/working/lora_deberta",
    num_train_epochs=6,                 # train for 6 epochs
    per_device_train_batch_size=16,     # batch size per gpu step
    gradient_accumulation_steps=2,      # effective batch size = 32
    learning_rate=2e-4,                 # lr for lora finetuning
    warmup_steps=50,                    # gradual lr warmup
    fp16=False,                         # disabled for better compatibility
    logging_steps=50,
    eval_strategy="epoch",              # eval after each epoch
    save_strategy="no",                 # no checkpoints needed
    report_to="wandb",                  # log metrics to wandb
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=42
)

wandb.init(
    project="23f1000362-t22026",
    name="model5-lora-deberta-losscurve",
    config={
        "lora_r": 16,
        "lora_alpha": 32,
        "lr": 2e-4,
        "epochs": 6,
        "base_model": "microsoft/deberta-v3-base"
    },
    tags=["milestone-5", "lora", "deberta", "loss-curve"]
)

trainer_deberta = Trainer(
    model=lora_deberta,
    args=args_deberta,
    train_dataset=hf_deberta_train,
    eval_dataset=hf_deberta_val,
    data_collator=deberta_collator,
    processing_class=deberta_tok,
    compute_metrics=compute_metrics
)

print("starting lora deberta finetuning...")
trainer_deberta.train()

lora_deberta.eval()
wandb.finish()

def rank_lora_deberta(row):
    inputs = deberta_tok(
        format_mcq(row),
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = lora_deberta(**inputs).logits[0]

    scores = {ID2LETTER[i]: logits[i].item() for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

deberta_preds = [rank_lora_deberta(row)[:3] for _, row in train.iterrows()]

map3_deberta = round(mapk(actuals, deberta_preds), 4)
acc_deberta  = top1_accuracy(actuals, deberta_preds)
f1_deberta   = top1_f1(actuals, deberta_preds)

print(f"lora deberta  MAP@3 : {map3_deberta}")
print(f"lora deberta — acc: {acc_deberta}, f1: {f1_deberta}")

run = wandb.init(
    project="23f1000362-t22026",
    name="model5-lora-deberta",
    config={
        "lora_r": 16,
        "lora_alpha": 32,
        "lr": 2e-4,
        "epochs": 6,
        "base_model": "microsoft/deberta-v3-base"
    },
    tags=["milestone-5", "lora", "deberta"]
)

wandb.log({
    "map3": map3_deberta,
    "accuracy": acc_deberta,
    "f1": f1_deberta
})

wandb.finish()

print("\ncell 8 complete - lora deberta trained and logged")

In [ ]:
results = {
    "tfidf": map3_tfidf,
    "word2vec": map3_w2v,
    "roberta_cosine": map3_roberta,
    "lora_single": map3_lora,
    "lora_weighted_ensemble": map3_ens_wt,
    "lora_deberta": map3_deberta,
}

rank_fns = {
    "tfidf": rank_tfidf,
    "word2vec": rank_w2v,
    "roberta_cosine": rank_roberta,
    "lora_single": rank_lora,
    "lora_weighted_ensemble": rank_lora_ensemble_weighted,
    "lora_deberta": rank_lora_deberta
}

print("MAP@3 comparison across all models:")
for k, v in results.items():
    print(f"  {k:25s}: {v}")

run = wandb.init(
    project="23f1000362-t22026",
    name="milestone5-final-comparison",
    tags=["milestone-5", "final"]
)
wandb.log({f"map3/{k}": v for k, v in results.items()})
wandb.finish()

def get_deberta_logits(row):
    inputs = deberta_tok(
        format_mcq(row),
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = lora_deberta(**inputs).logits[0]

    return logits.cpu().numpy()

# combine roberta and deberta predictions
def rank_cross_arch_ensemble(row):
    roberta_logits = np.array([get_lora_logits(m, row) for m in lora_models])

    roberta_weights_norm = np.array(lora_weights) / sum(lora_weights)
    roberta_avg = np.average(roberta_logits, axis=0, weights=roberta_weights_norm)

    deberta_logits = get_deberta_logits(row)

    # weighted fusion of both models
    final = 0.65 * roberta_avg + 0.35 * deberta_logits

    scores = {ID2LETTER[i]: final[i] for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

cross_preds = [rank_cross_arch_ensemble(row)[:3] for _, row in train.iterrows()]
map3_cross = round(mapk(actuals, cross_preds), 4)
acc_cross  = top1_accuracy(actuals, cross_preds)
f1_cross   = top1_f1(actuals, cross_preds)

print(f"cross-architecture ensemble (roberta+deberta)  MAP@3 : {map3_cross} | acc: {acc_cross}, f1: {f1_cross}")

results["cross_arch_ensemble"] = map3_cross
rank_fns["cross_arch_ensemble"] = rank_cross_arch_ensemble

run = wandb.init(
    project="23f1000362-t22026",
    name="model6-cross-architecture-ensemble",
    config={
        "components": ["lora_roberta_3seed", "lora_deberta"],
        "weights": [0.65, 0.35]
    },
    tags=["milestone-5", "ensemble", "cross-architecture"]
)

wandb.log({
    "map3": map3_cross,
    "accuracy": acc_cross,
    "f1": f1_cross
})
wandb.finish()

# pick best model for final submission
best = max(results, key=lambda x: results[x])
print(f"\nbest model: {best} (MAP@3 = {results[best]})")

best_fn = rank_fns[best]

# test_preds = [" ".join(best_fn(row)[:3]) for _, row in test.iterrows()]
# submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
# submission.to_csv("submission.csv", index=False)
# print("submission.csv saved")
# submission.head(5)

In [ ]:
SUBMIT_FN = rank_lora_ensemble_weighted   # clean 3-seed prob-avg ensemble — try this for >0.76
# SUBMIT_FN = rank_lora                    # single seed-42 — your proven 0.757 fallback

test_preds = [" ".join(SUBMIT_FN(row)[:3]) for _, row in test.iterrows()]
submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv saved using {SUBMIT_FN.__name__}")
submission.head(5)